In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Nishan\\Downloads\\datascienceproject (1)\\datascienceproject\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Nishan\\Downloads\\datascienceproject (1)\\datascienceproject'

In [5]:
import pandas as pd
df = pd.read_csv('artifacts/data_ingestion/winequality-red.csv')

In [6]:
df.dtypes

fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

In [ ]:
# entity/config_entity.py
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: Path
    all_schema: dict
    target: dict

In [15]:
# config/configuration.py
from src.datascienceproject.constants import CONFIG_FILE_PATH
from src.datascienceproject.constants import PARAMS_FILE_PATH
from src.datascienceproject.constants import SCHEMA_FILE_PATH

from src.datascienceproject.utils.common import read_yaml, create_directories

class ConfigurationManager:
    
	def __init__(self,
			  	config_file_path = CONFIG_FILE_PATH,
				params_file_path = PARAMS_FILE_PATH,
				schema_file_path = SCHEMA_FILE_PATH):
		
		self.config = read_yaml(config_file_path)
		self.params = read_yaml(params_file_path)
		self.schema = read_yaml(schema_file_path)

		create_directories([self.config.artifacts_root])

	def get_data_validation_config(self):

		config = self.config.data_validation
		schema = self.schema.COLUMNS
		target = self.schema.TARGET_COLUMN

		create_directories([config.root_dir])

		data_validation_config = DataValidationConfig(
			root_dir = config.root_dir,
			unzip_data_dir = config.unzip_data_dir,
			STATUS_FILE = config.STATUS_FILE,
			all_schema = schema,
			target = target
		)

		return data_validation_config

In [ ]:
# components/data_validation.py
import os
from src.datascienceproject import logger
import pandas as pd

class DataValidation:
	def __init__(self, config: DataValidationConfig):
		self.config = config
		
	def validate_all_columns(self) -> bool:

		try:
			# Checking validation status
			validation_status = True
			df = pd.read_csv(self.config.unzip_data_dir)
			schema = self.config.all_schema
			target_col = list(self.config.target.keys())[0]

			all_columns = [i for i in df.columns if i != target_col]
			for col in all_columns:
				if str(df[col].dtype ) == self.config.all_schema[col]:
					pass
				else:
					validation_status = False

			# Writing validation status to txt file
			with open(self.config.STATUS_FILE, 'w') as f:
				f.write(str(validation_status))

			logger.info(f"Data validation completed and file saved at: {self.config.STATUS_FILE}")

		except Exception as e:
			logger.exception(e)


In [27]:
# pipeline/data_validation_pipeline.py
config_obj = ConfigurationManager()
data_validation_config = config_obj.get_data_validation_config()
data_validation = DataValidation(data_validation_config)
data_validation.validate_all_columns()

2026-05-26 11:44:00,962 : INFO : YAML file config\config.yaml loaded successfully
2026-05-26 11:44:00,963 : INFO : YAML file params.yaml loaded successfully
2026-05-26 11:44:00,966 : INFO : YAML file schema.yaml loaded successfully
2026-05-26 11:44:00,966 : INFO : Created directory at artifacts
2026-05-26 11:44:00,967 : INFO : Created directory at artifacts/data_validation
2026-05-26 11:44:00,975 : INFO : Data validation completed and file saved at: artifacts/data_validation/status.txt


'quality'

In [ ]:
# main.py


In [13]:
data_validation_config.all_schema

ConfigBox({'fixed acidity': 'float64', 'volatile acidity': 'float64', 'citric acid': 'float64', 'residual sugar': 'float64', 'chlorides': 'float64', 'free sulfur dioxide': 'float64', 'total sulfur dioxide': 'float64', 'density': 'float64', 'pH': 'float64', 'sulphates': 'float64', 'alcohol': 'float64'})

'float64'

In [18]:
data_validation_config.all_schema['fixed acidity']

'float64'

In [19]:
data_validation_config

DataValidationConfig(root_dir='artifacts/data_validation', unzip_data_dir='artifacts/data_ingestion/winequality-data.csv', STATUS_FILE='artifacts/data_validation/status.txt', all_schema=ConfigBox({'fixed acidity': 'float64', 'volatile acidity': 'float64', 'citric acid': 'float64', 'residual sugar': 'float64', 'chlorides': 'float64', 'free sulfur dioxide': 'float64', 'total sulfur dioxide': 'float64', 'density': 'float64', 'pH': 'float64', 'sulphates': 'float64', 'alcohol': 'float64'}))